# 01 — EDA & Paper Replication
## Replicating DCCore.pdf Monte Carlo Results

This notebook replicates the core Monte Carlo simulation from the original paper,
validating our implementation against the published TCO figures for 10 global datacenter locations.

**Key outputs:**
- TCO distributions for all 10 locations
- Risk metrics (VaR, CVaR, CV) comparison with paper
- Sensitivity analysis: which variables drive TCO most?

**Paper reference:** DCCore.pdf — Monte Carlo simulation with 10,000 iterations per location,
triangular distributions for CapEx/OpEx/PUE, 7% discount rate, 10-year horizon.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.location_profiles import load_locations
from src.tco.components import TCOParams, compute_tco, compute_annual_power_cost
from src.tco.monte_carlo import run_static_mc, run_all_locations
from src.risk.metrics import compute_risk_metrics, risk_premium

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

locations = load_locations()
print(f'Loaded {len(locations)} locations')
for key, loc in locations.items():
    print(f'  {loc.name}: {loc.tier}, {loc.currency}, baseline TCO={loc.mean_tco_10yr_millions}M')

## 1. Run Static Monte Carlo Simulation (10K iterations)

In [ ]:
results = run_all_locations(locations, n_simulations=10000, horizon_years=10, seed=42)

# Build summary table
summary_rows = []
for key, res in results.items():
    loc = locations[key]
    rm = compute_risk_metrics(res.tco_distribution)
    summary_rows.append({
        'Location': loc.name,
        'Tier': loc.tier,
        'Currency': loc.currency,
        'Mean TCO (M)': round(rm.mean, 1),
        'Paper TCO (M)': loc.mean_tco_10yr_millions,
        'Std (M)': round(rm.std, 1),
        'CV': round(rm.cv, 4),
        'Paper CV': loc.cv,
        'VaR 95% (M)': round(rm.var_95, 1),
        'CVaR 95% (M)': round(rm.cvar_95, 1),
        'P5 (M)': round(rm.p5, 1),
        'P95 (M)': round(rm.p95, 1),
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Mean TCO (M)')
summary_df

## 2. TCO Distribution Plots

In [ ]:
# Nordic vs US vs Traditional comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tier_groups = {
    'Nordic (EUR)': ['boden_sweden', 'kristiansand_norway', 'iceland_reykjanes'],
    'US Secondary (USD)': ['evanston_wyoming', 'salt_lake_city_utah', 'des_moines_iowa', 'florence_south_carolina'],
    'Traditional/Emerging': ['atlanta_georgia', 'johor_malaysia', 'sines_portugal'],
}

for ax, (tier_name, loc_keys) in zip(axes, tier_groups.items()):
    for key in loc_keys:
        dist = results[key].tco_distribution
        ax.hist(dist, bins=50, alpha=0.5, label=f'{locations[key].name}\nμ={np.mean(dist):.0f}M')
    ax.set_title(tier_name, fontsize=13)
    ax.set_xlabel('10-Year TCO (Millions)')
    ax.legend(fontsize=8)

fig.suptitle('TCO Distributions by Market Tier', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Full comparison box plot
fig, ax = plt.subplots(figsize=(14, 7))

sorted_keys = sorted(results.keys(), key=lambda k: results[k].mean)
box_data = [results[k].tco_distribution for k in sorted_keys]
labels = [locations[k].name for k in sorted_keys]

bp = ax.boxplot(box_data, labels=labels, vert=True, patch_artist=True, showfliers=False)

tier_colors = {'nordic': '#3498db', 'us_secondary': '#2ecc71', 'traditional': '#e74c3c', 'emerging': '#f39c12'}
for i, key in enumerate(sorted_keys):
    bp['boxes'][i].set_facecolor(tier_colors.get(locations[key].tier, 'gray'))
    bp['boxes'][i].set_alpha(0.7)

ax.set_ylabel('10-Year TCO (Millions)', fontsize=12)
ax.set_title('TCO Comparison Across All 10 Locations (10K Monte Carlo Simulations)', fontsize=14)
plt.xticks(rotation=35, ha='right')

from matplotlib.patches import Patch
legend_items = [Patch(facecolor=c, label=t.replace('_', ' ').title()) for t, c in tier_colors.items()]
ax.legend(handles=legend_items, loc='upper left')

plt.tight_layout()
plt.show()

## 3. Risk Metrics Comparison

In [ ]:
# Risk-return scatter: Mean TCO vs Coefficient of Variation
fig, ax = plt.subplots(figsize=(10, 7))

for key in results:
    rm = compute_risk_metrics(results[key].tco_distribution)
    color = tier_colors.get(locations[key].tier, 'gray')
    ax.scatter(rm.mean, rm.cv, s=150, c=color, edgecolors='black', linewidth=0.5, zorder=5)
    ax.annotate(locations[key].name, (rm.mean, rm.cv),
                textcoords='offset points', xytext=(8, 5), fontsize=8)

ax.set_xlabel('Mean 10-Year TCO (Millions)', fontsize=12)
ax.set_ylabel('Coefficient of Variation (Risk)', fontsize=12)
ax.set_title('Risk-Return Profile: Lower-Left is Optimal', fontsize=14)
ax.legend(handles=legend_items, loc='upper right')
plt.tight_layout()
plt.show()

## 4. Sensitivity Analysis
Which variables have the largest impact on TCO?

In [ ]:
# Sensitivity: +/- 20% perturbation on key variables for Boden, Sweden
loc = locations['boden_sweden']
base_params = TCOParams(
    capex_millions=loc.capex_millions[1],
    power_cost_mwh=loc.power_cost_mwh[1],
    pue=loc.pue[1],
)
base_tco = compute_tco(base_params)

variables = {
    'Power Cost/MWh': ('power_cost_mwh', loc.power_cost_mwh[1]),
    'CapEx': ('capex_millions', loc.capex_millions[1]),
    'PUE': ('pue', loc.pue[1]),
    'Utilization': ('utilization', 0.75),
    'Staffing Cost': ('staffing_annual_millions', 5.0),
    'Maintenance %': ('maintenance_pct', 0.03),
    'Insurance': ('insurance_annual_millions', 2.0),
}

sensitivities = {}
for var_name, (param_name, base_val) in variables.items():
    for direction, mult in [('-20%', 0.8), ('+20%', 1.2)]:
        params_dict = {
            'capex_millions': loc.capex_millions[1],
            'power_cost_mwh': loc.power_cost_mwh[1],
            'pue': loc.pue[1],
        }
        params_dict[param_name] = base_val * mult
        shifted_tco = compute_tco(TCOParams(**params_dict))
        pct_change = (shifted_tco - base_tco) / base_tco * 100
        sensitivities[f'{var_name} {direction}'] = pct_change

# Tornado chart
fig, ax = plt.subplots(figsize=(10, 6))
items = sorted(sensitivities.items(), key=lambda x: abs(x[1]))
names, vals = zip(*items)
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in vals]
ax.barh(names, vals, color=colors)
ax.set_xlabel('TCO Change (%)')
ax.set_title(f'Sensitivity Analysis: {loc.name} (±20% Perturbation)', fontsize=13)
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 5. Key Findings

| Finding | Detail |
|---------|--------|
| **Nordic Dominance** | Boden, Sweden consistently lowest TCO across simulations |
| **Power Cost is King** | Highest sensitivity variable — confirms paper's conclusion |
| **Risk Profile** | Nordic locations show lowest CV (best risk-adjusted returns) |
| **Geographic Arbitrage** | 35-74% cost gap between Nordic and traditional hubs |